In [1]:
# # CPU версия парсера 
# import os
# import sys
# import glob
# import time
# import chromadb
# from concurrent.futures import ProcessPoolExecutor
# from sentence_transformers import SentenceTransformer
# from dotenv import load_dotenv

# project_root = os.path.abspath('..')
# src_path = os.path.join(project_root, 'src')
# if src_path not in sys.path:
#     sys.path.append(src_path)

# from hypothesis_factory.ingestion import PyMuPDFReader

# load_dotenv()
# API_KEY = os.getenv("YANDEX_API_KEY")
# FOLDER_ID = os.getenv("YANDEX_FOLDER_ID")
# if not API_KEY or not FOLDER_ID:
#     raise ValueError("Ключи не найдены! Проверь файл .env")


# def parse_one(pdf_path: str):
#     """Функция для отдельного процесса: парсит один PDF и возвращает чанки + время"""
#     reader = PyMuPDFReader(extract_tables=False)
#     t0 = time.time()
#     chunks = reader.read(pdf_path)
#     dt = time.time() - t0
#     return pdf_path, chunks, dt


# if __name__ == "__main__":
#     print("Загрузка модели эмбеддингов (GPU)...")
#     embed_model = SentenceTransformer('intfloat/multilingual-e5-large', device='cuda')

#     print("Инициализация ChromaDB...")
#     client = chromadb.Client()
#     try:
#         client.delete_collection("materials_db_notebook")
#     except Exception:
#         pass
#     collection = client.create_collection("materials_db_notebook")

#     raw_data_dir = os.path.join(project_root, "data", "raw")
#     pdf_files = glob.glob(os.path.join(raw_data_dir, "*.pdf"))
#     if not pdf_files:
#         raise ValueError(f"PDF файлы не найдены в папке {raw_data_dir}!")

#     print(f"\nНачинаем парсинг {len(pdf_files)} документов (параллельно, {os.cpu_count()} ядер)...")
#     t_start = time.time()

#     documents, metadatas, ids = [], [], []

#     with ProcessPoolExecutor(max_workers=os.cpu_count()) as executor:
#         for pdf_path, chunks, dt in executor.map(parse_one, pdf_files):
#             fname = os.path.basename(pdf_path)
#             flag = "🟢" if dt < 5 else ("🟡" if dt < 20 else "🔴")
#             print(f" {flag} {fname}: {dt:.1f} сек, {len(chunks)} чанков")

#             for chunk in chunks:
#                 documents.append(chunk.text)
#                 ids.append(chunk.chunk_id)
#                 metadatas.append({
#                     "source_id": chunk.metadata.source_id,
#                     "title": chunk.metadata.title,
#                     "source_type": chunk.metadata.source_type
#                 })

#     print(f"\nПарсинг завершён за {time.time() - t_start:.1f} сек.")
#     print(f"Извлечено {len(documents)} фрагментов. Начинаем векторизацию...")

#     t0 = time.time()
#     embeddings = embed_model.encode(
#         documents, batch_size=64, show_progress_bar=True
#     ).tolist()
#     print(f"Векторизация заняла {time.time() - t0:.1f} сек")

#     collection.add(
#         documents=documents,
#         embeddings=embeddings,
#         metadatas=metadatas,
#         ids=ids
#     )

#     print("\nБаза знаний успешно заполнена реальными статьями и готова к поиску!")

In [2]:
# GPU версия парсера
import os
import sys
import glob
import time
import torch
import chromadb
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv

project_root = os.path.abspath('..')
src_path = os.path.join(project_root, 'src')
if src_path not in sys.path:
    sys.path.append(src_path)

from hypothesis_factory.ingestion_gpu import GpuDoclingReader

load_dotenv()
API_KEY = os.getenv("YANDEX_API_KEY")
FOLDER_ID = os.getenv("YANDEX_FOLDER_ID")
if not API_KEY or not FOLDER_ID:
    raise ValueError("Ключи не найдены! Проверь файл .env")

# ==========================================================
# 0. ДИАГНОСТИКА: проверяем, что PyTorch реально видит GPU
# ==========================================================
print("=" * 60)
print("ПРОВЕРКА GPU")
print("=" * 60)
cuda_ok = torch.cuda.is_available()
print(f"torch.cuda.is_available(): {cuda_ok}")
if cuda_ok:
    print(f"Устройство: {torch.cuda.get_device_name(0)}")
    print(f"Свободно VRAM: {torch.cuda.mem_get_info()[0] / 1024**3:.1f} GB "
          f"из {torch.cuda.mem_get_info()[1] / 1024**3:.1f} GB")
else:
    print("    CUDA НЕДОСТУПНА. Проверьте установку torch с поддержкой CUDA:")
    print("    pip install torch --index-url https://download.pytorch.org/whl/cu121")
    print("    (версию cu121 подберите под вашу версию CUDA driver)")

DEVICE = "cuda" if cuda_ok else "cpu"

# ==========================================================
# 1. Модель эмбеддингов
# ==========================================================
print("\nЗагрузка модели эмбеддингов...")
embed_model = SentenceTransformer('intfloat/multilingual-e5-large', device=DEVICE)
print(f"Эмбеддинг-модель загружена на: {embed_model.device}")

# ==========================================================
# 2. ChromaDB
# ==========================================================
print("\nИнициализация ChromaDB...")
client = chromadb.Client()
try:
    client.delete_collection("materials_db_notebook")
except Exception:
    pass
collection = client.create_collection("materials_db_notebook")

# ==========================================================
# 3. Docling reader с ЯВНЫМ указанием GPU
# ==========================================================
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import (
    PdfPipelineOptions,
    AcceleratorOptions,
    AcceleratorDevice,
)
from docling.datamodel.base_models import InputFormat
from docling.chunking import HierarchicalChunker

print("\nНастройка Docling с GPU-ускорителем...")

accelerator_options = AcceleratorOptions(
    num_threads=6,  # CPU слабый — не даём Docling плодить лишние потоки
    device=AcceleratorDevice.CUDA if cuda_ok else AcceleratorDevice.CPU,
)

pipeline_options = PdfPipelineOptions()
pipeline_options.accelerator_options = accelerator_options
pipeline_options.do_ocr = False              # оставляем — в PDF могут быть сканы
pipeline_options.do_table_structure = False  # если таблицы не нужны, можно поставить False

_converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)
_chunker = HierarchicalChunker()

# Подменяем внутренние объекты у уже готового ридера,
# чтобы не трогать файл ingestion_gpu.py
reader = GpuDoclingReader()
reader.converter = _converter
reader.chunker = _chunker

print(f"Docling будет использовать устройство: {accelerator_options.device}")

# ==========================================================
# 4. Парсинг с замером времени по каждому файлу
# ==========================================================
raw_data_dir = os.path.join(project_root, "data", "raw")
pdf_files = glob.glob(os.path.join(raw_data_dir, "*.pdf"))
if not pdf_files:
    raise ValueError(f"PDF файлы не найдены в папке {raw_data_dir}!")

print(f"\nНачинаем парсинг {len(pdf_files)} документов...")
documents = []
metadatas = []
ids = []

for pdf_path in pdf_files:
    fname = os.path.basename(pdf_path)
    t0 = time.time()
    chunks = reader.read(pdf_path)
    dt = time.time() - t0

    speed_flag = "🟢" if dt < 15 else ("🟡" if dt < 60 else "🔴")
    print(f" {speed_flag} {fname}: {dt:.1f} сек, {len(chunks)} чанков")

    for chunk in chunks:
        documents.append(chunk.text)
        ids.append(chunk.chunk_id)
        metadatas.append({
            "source_id": chunk.metadata.source_id,
            "title": chunk.metadata.title,
            "source_type": chunk.metadata.source_type
        })

print(f"\nИзвлечено {len(documents)} фрагментов. Начинаем векторизацию...")

# ==========================================================
# 5. Эмбеддинги
# ==========================================================
t0 = time.time()
embeddings = embed_model.encode(
    documents,
    batch_size=64 if cuda_ok else 16,  # на GPU можно батч побольше
    show_progress_bar=True,
).tolist()
print(f"Векторизация заняла {time.time() - t0:.1f} сек")

collection.add(
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas,
    ids=ids
)

print("\nБаза знаний успешно заполнена реальными статьями и готова к поиску!")

ПРОВЕРКА GPU
torch.cuda.is_available(): True
Устройство: NVIDIA A100 80GB PCIe MIG 2g.20gb
Свободно VRAM: 19.3 GB из 19.5 GB

Загрузка модели эмбеддингов...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Эмбеддинг-модель загружена на: cuda:0

Инициализация ChromaDB...

Настройка Docling с GPU-ускорителем...
Docling будет использовать устройство: AcceleratorDevice.CUDA

Начинаем парсинг 3 документов...


Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

 🟢 geokniga-tehnologiyaobogashcheniyapoleznyhiskopaemyh.pdf: 6.8 сек, 238 чанков
 🟡 geokniga-flotacionnye-metody-obogashcheniya_0.pdf: 45.0 сек, 2385 чанков
 🟡 geokniga-metallurgiya-blagorodnyh-metallov_0.pdf: 31.5 сек, 2358 чанков

Извлечено 4981 фрагментов. Начинаем векторизацию...


Batches:   0%|          | 0/78 [00:00<?, ?it/s]

Векторизация заняла 65.3 сек

База знаний успешно заполнена реальными статьями и готова к поиску!


In [12]:
import requests

def generate_hypothesis_pipeline(problem, constraints):
    print(f"ЗАПУСК ПАЙПЛАЙНА")
    print(f"Проблема: {problem}")
    print(f"Ограничения: {constraints}\n")
    
    print("Ищем релевантный контекст в базе...")

    query_text = f"{problem}. Ограничения: {constraints}"
    query_vector = embed_model.encode(query_text).tolist()
    
    results = collection.query(
        query_embeddings=[query_vector],
        n_results=2
    )
    
    retrieved_context = "\n- ".join(results['documents'][0])
    print("Найденный контекст:")
    print(f"- {retrieved_context}\n")
    
    print("Отправка промпта в Yandex AI Studio...")
    url = "https://llm.api.cloud.yandex.net/foundationModels/v1/completion"
    
    system_prompt = (
        "Ты — научный ассистент НИИ. Твоя задача — сгенерировать проверяемую научную гипотезу "
        "строго на основе предоставленного контекста. "
        "Ответ структурируй: 1. Гипотеза 2. Обоснование (с опорой на контекст) 3. Риски 4. Ожидаемый эффект."
    )
    
    user_prompt = f"Проблема: {problem}\nОграничения: {constraints}\n\nКонтекст из базы знаний:\n{retrieved_context}"
    
    payload = {
        "modelUri": f"gpt://{FOLDER_ID}/yandexgpt/latest",
        "completionOptions": {"stream": False, "temperature": 0.2, "maxTokens": 2000},
        "messages": [
            {"role": "system", "text": system_prompt},
            {"role": "user", "text": user_prompt}
        ]
    }
    
    headers = {"Content-Type": "application/json", "Authorization": f"Api-Key {API_KEY}"}
    response = requests.post(url, headers=headers, json=payload)
    
    if response.status_code == 200:
        answer = response.json()['result']['alternatives'][0]['message']['text']
        print("Ответ модели:\n")
        print("="*50)
        print(answer)
        print("="*50)
    else:
        print(f"Ошибка API {response.status_code}: {response.text}")

target_problem = "Уменьшения вредного влияния на процесс цианирования руды сурьмянистых и мышьяковистых минералов"
user_constraints = "Бюджет ограничен, необходимо избегать дорогих компонентов"

# Запуск
generate_hypothesis_pipeline(target_problem, user_constraints)

ЗАПУСК ПАЙПЛАЙНА
Проблема: Уменьшения вредного влияния на процесс цианирования руды сурьмянистых и мышьяковистых минералов
Ограничения: Бюджет ограничен, необходимо избегать дорогих компонентов

Ищем релевантный контекст в базе...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Найденный контекст:
- Для уменьшения вредного влияния на процесс цианирования ру -ды сурьмянистых и мышьяковистых минералов следует :
- - -снизить до минимума концентрацию защитной щелочи
- -максимально  быстро превращать  вредные тиосоли и сульфид -
- ионы в относительно безвредные роданид -ионы введением в процесс растворимых  солей  свинца -азотно -кислого  или  уксусно -кислого . Вместо указанных  солей свинца  можно  применять  более дешевый

Отправка промпта в Yandex AI Studio...
Ошибка API 403: {"error":{"grpcCode":7,"httpCode":403,"message":"Permission denied","httpStatus":"Forbidden","details":[]}}


In [8]:
# Тест оркестра


# ==========================================================
# 1. Импорты и настройка
# ==========================================================
import logging

# Строгие импорты из пакета hypothesis_factory
from hypothesis_factory.base import BusinessRequest, DocumentChunk, DocumentMetadata
from hypothesis_factory.generator import YandexGPTGenerator
from hypothesis_factory.critic import YandexGPTCritic
from hypothesis_factory.pipeline import HypothesisRefinementPipeline

# Настроим логирование, чтобы видеть процесс общения Генератора и Критика прямо в ячейке
logging.getLogger().setLevel(logging.INFO)

# ==========================================================
# 2. Формируем бизнес-запрос
# ==========================================================
print("=== ШАГ 1: Формирование бизнес-запроса ===")
request = BusinessRequest(
    target_kpi="Повысить предел прочности сплава на 15% при сохранении пластичности.",
    constraints=[
        "Запрещено использование токсичных или радиоактивных элементов",
        "Бюджет на экспериментальную партию не более 500 000 рублей",
        "Технология должна быть применима на стандартном прокатном стане"
    ]
)
print(f"Цель: {request.target_kpi}")
print(f"Ограничения: {request.constraints}\n")

# ==========================================================
# 3. Поиск релевантного контекста (Retriever)
# ==========================================================
print("=== ШАГ 2: Векторный поиск по базе знаний (ChromaDB) ===")
# Формируем строку запроса для поиска на основе цели и ограничений
query_text = f"{request.target_kpi} Ограничения: {', '.join(request.constraints)}"
query_embedding = embed_model.encode([query_text]).tolist()

# Ищем топ-7 самых релевантных кусков текста
results = collection.query(
    query_embeddings=query_embedding,
    n_results=7 
)

context_chunks = []
if results['ids'] and results['ids'][0]:
    for i in range(len(results['ids'][0])):
        chunk_id = results['ids'][0][i]
        text = results['documents'][0][i]
        meta = results['metadatas'][0][i]

        # Преобразуем данные из ChromaDB в наши Pydantic контракты
        metadata = DocumentMetadata(
            source_id=meta.get("source_id", f"unknown_doc_{i}"),
            source_type=meta.get("source_type", "article"),
            title=meta.get("title", "Неизвестный источник")
        )
        context_chunks.append(DocumentChunk(chunk_id=chunk_id, text=text, metadata=metadata))

print(f"Найдено релевантных фрагментов: {len(context_chunks)}\n")

# ==========================================================
# 4. Запуск Оркестратора (Self-Correction Loop) с закрытием сессий
# ==========================================================
print("=== ШАГ 3: Запуск LLM-пайплайна (Генератор + Критик) ===")

# Используем контекстные менеджеры, чтобы гарантировать закрытие API-сессий
with YandexGPTGenerator() as generator, YandexGPTCritic() as critic:
    
    # Собираем оркестратор
    pipeline = HypothesisRefinementPipeline(
        generator=generator, 
        critic=critic,
        min_overall_score=7.0, 
        max_iterations=3
    )

    # Запускаем цикл (пока мы внутри with, API-соединения активны)
    final_hypotheses = pipeline.run_loop(request=request, context=context_chunks)

# Как только код выходит из блока with, автоматически вызываются generator.close() и critic.close()
print("API-сессии успешно закрыты.")

# ==========================================================
# 5. Красивый вывод результатов
# ==========================================================
print("\n" + "="*80)
print(f" ИТОГОВЫЙ ОТЧЕТ: СГЕНЕРИРОВАНО ГИПОТЕЗ - {len(final_hypotheses)}")
print("="*80 + "\n")

for i, hyp in enumerate(final_hypotheses, 1):
    status = " ВАЛИДНА" if hyp.overall_score >= pipeline.min_overall_score else " НИЖЕ ПОРОГА"
    
    print(f"[{i}] {hyp.title.upper()} ({status})")
    print(f"Рейтинг: {hyp.overall_score}/10 (Новизна: {hyp.novelty_score}, Реализуемость: {hyp.feasibility_score})")
    print(f"Суть: {hyp.text}")
    print(f"Обоснование: {hyp.reasoning}")
    print(f"Механизм: {hyp.mechanism}")
    
    if hyp.technical_risks:
        print("Технические риски:")
        for risk in hyp.technical_risks:
            print(f"  - {risk}")
            
    if hyp.economic_risks:
        print("Экономические риски:")
        for risk in hyp.economic_risks:
            print(f"  - {risk}")
            
    if hyp.source_refs:
        print(f"Источники: {', '.join(hyp.source_refs)}")
    else:
        print("Источники: Базовые знания модели (Вне контекста)")
        
    print("-" * 80 + "\n")

=== ШАГ 1: Формирование бизнес-запроса ===
Цель: Повысить предел прочности сплава на 15% при сохранении пластичности.
Ограничения: ['Запрещено использование токсичных или радиоактивных элементов', 'Бюджет на экспериментальную партию не более 500 000 рублей', 'Технология должна быть применима на стандартном прокатном стане']

=== ШАГ 2: Векторный поиск по базе знаний (ChromaDB) ===


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Найдено релевантных фрагментов: 7

=== ШАГ 3: Запуск LLM-пайплайна (Генератор + Критик) ===


API call failed on attempt 1: Error code: 403 - {'error': {'message': 'rpc error: code = PermissionDenied desc = Permission denied', 'type': 'permission_error'}}
Max retries exceeded. Total attempts: 1, Last error: Error code: 403 - {'error': {'message': 'rpc error: code = PermissionDenied desc = Permission denied', 'type': 'permission_error'}}
Критический сбой при генерации гипотез: Error code: 403 - {'error': {'message': 'rpc error: code = PermissionDenied desc = Permission denied', 'type': 'permission_error'}}
Генератор не смог предложить базовый набор гипотез.



 ИТОГОВЫЙ ОТЧЕТ: СГЕНЕРИРОВАНО ГИПОТЕЗ - 0



In [9]:
# Офлайн тест


# ==========================================================
# 1. Импорты и настройка
# ==========================================================
import logging
from typing import List

from hypothesis_factory.base import BusinessRequest, DocumentChunk, DocumentMetadata, Hypothesis, HypothesisList
from hypothesis_factory.critic import EvaluationResult
from hypothesis_factory.pipeline import HypothesisRefinementPipeline

logging.getLogger().setLevel(logging.INFO)

# ==========================================================
# 2. Создаем Mock-классы (Заглушки API)
# ==========================================================

class FakeCompletions:
    def create(self, **kwargs):
        # Эта функция вызовется внутри pipeline._refine_hypotheses
        print("\n[MOCK API] Генератор получил запрос на ДОРАБОТКУ гипотезы!")
        
        # Эмулируем, что LLM поняла критику и исправила гипотезу
        refined_hyp = Hypothesis(
            id="mock_2", # ID должен совпадать с исходным
            title="[ИСПРАВЛЕНО] Добавление 0.05% графена",
            text="Снижена концентрация графена для исключения выгорания и снижения цены.",
            mechanism="Микродозирование нанотрубок.",
            reasoning="Исправлены экономические и физические риски.",
            source_refs=["chunk_mock_1"],
            novelty_score=0.0, feasibility_score=0.0, overall_score=0.0
        )
        return HypothesisList(
            preliminary_analysis="Я учел замечания критика и снизил концентрацию графена.",
            hypotheses=[refined_hyp]
        )

class FakeChat:
    def __init__(self):
        self.completions = FakeCompletions()

class FakeClient:
    def __init__(self):
        self.chat = FakeChat()

class MockGenerator:
    def __init__(self):
        self.client = FakeClient()
        self.model_uri = "mock://yandexgpt"

    def generate(self, request: BusinessRequest, context: List[DocumentChunk]) -> List[Hypothesis]:
        print("\n[MOCK API] Вызван метод первичной генерации гипотез...")
        # Генерируем 2 фейковые гипотезы: одну "хорошую", вторую "проблемную"
        hyp1 = Hypothesis(
            id="mock_1",
            title="Легирование титаном (Хорошая)",
            text="Добавление 1% титана в сплав.",
            mechanism="Образование карбидов.",
            reasoning="Титан повышает прочность.",
            source_refs=["chunk_mock_1"]
        )
        hyp2 = Hypothesis(
            id="mock_2",
            title="Добавление 5% графена (Спорная)",
            text="Внедрение большого количества графеновых нанотрубок.",
            mechanism="Углеродный каркас.",
            reasoning="Графен очень прочный.",
            source_refs=["chunk_mock_1"]
        )
        return [hyp1, hyp2]


class MockCritic:
    def _score_single_hypothesis(self, hyp: Hypothesis, context: List[DocumentChunk]) -> EvaluationResult:
        print(f"[MOCK API] Критик оценивает: {hyp.title}")
        
        if "Хорошая" in hyp.title:
            # Сразу проходит порог (8.0 + 8.0) / 2 = 8.0
            return EvaluationResult(
                novelty_score=8.0, feasibility_score=8.0,
                technical_risks=["Сложность контроля температуры"], economic_risks=["Удорожание на 5%"],
                is_valid=True
            )
        elif "[ИСПРАВЛЕНО]" in hyp.title:
            # Исправленная гипотеза тоже проходит (7.0 + 8.0) / 2 = 7.5
            return EvaluationResult(
                novelty_score=7.0, feasibility_score=8.0,
                technical_risks=["Требуется точное диспергирование"], economic_risks=["Вписывается в бюджет"],
                is_valid=True
            )
        else:
            # Спорная гипотеза проваливается (9.0 + 3.0) / 2 = 6.0 (ниже порога)
            return EvaluationResult(
                novelty_score=9.0, feasibility_score=3.0,
                technical_risks=["Графен просто сгорит в расплаве стали"], economic_risks=["Цена в 100 раз выше бюджета"],
                is_valid=True 
            )

# ==========================================================
# 3. Подготовка тестовых данных
# ==========================================================
request = BusinessRequest(
    target_kpi="Тестовая цель для оффлайн проверки оркестратора.",
    constraints=["Без затрат API"]
)
mock_metadata = DocumentMetadata(source_id="mock_doc", source_type="article", title="Тестовая статья")
context_chunks = [DocumentChunk(chunk_id="chunk_mock_1", text="Тестовый текст.", metadata=mock_metadata)]

# ==========================================================
# 4. Запуск пайплайна (Offline)
# ==========================================================
print("=== СТАРТ ТЕСТОВОГО ПРОГОНА ===")

mock_gen = MockGenerator()
mock_crit = MockCritic()

pipeline = HypothesisRefinementPipeline(
    generator=mock_gen, 
    critic=mock_crit,
    min_overall_score=7.0, # Порог 7.0 
    max_iterations=2       # Даем 2 попытки
)

# Запускаем!
final_hypotheses = pipeline.run_loop(request=request, context=context_chunks)

# ==========================================================
# 5. Вывод результатов
# ==========================================================
print("\n" + "="*80)
print(f" ИТОГОВЫЙ ОТЧЕТ: СГЕНЕРИРОВАНО ГИПОТЕЗ - {len(final_hypotheses)}")
print("="*80 + "\n")

for i, hyp in enumerate(final_hypotheses, 1):
    print(f"[{i}] {hyp.title}")
    print(f"Балл: {hyp.overall_score}/10 (Новизна: {hyp.novelty_score}, Реализуемость: {hyp.feasibility_score})")
    print(f"Риски: {hyp.technical_risks[0]} | {hyp.economic_risks[0]}")
    print("-" * 50)

=== СТАРТ ТЕСТОВОГО ПРОГОНА ===

[MOCK API] Вызван метод первичной генерации гипотез...
[MOCK API] Критик оценивает: Легирование титаном (Хорошая)
[MOCK API] Критик оценивает: Добавление 5% графена (Спорная)

[MOCK API] Генератор получил запрос на ДОРАБОТКУ гипотезы!
[MOCK API] Критик оценивает: [ИСПРАВЛЕНО] Добавление 0.05% графена

 ИТОГОВЫЙ ОТЧЕТ: СГЕНЕРИРОВАНО ГИПОТЕЗ - 2

[1] Легирование титаном (Хорошая)
Балл: 8.0/10 (Новизна: 8.0, Реализуемость: 8.0)
Риски: Сложность контроля температуры | Удорожание на 5%
--------------------------------------------------
[2] [ИСПРАВЛЕНО] Добавление 0.05% графена
Балл: 7.5/10 (Новизна: 7.0, Реализуемость: 8.0)
Риски: Требуется точное диспергирование | Вписывается в бюджет
--------------------------------------------------
